In [ ]:
import random
import csv

# ──────────────────────────────────────────
# STRUCTURE ABR
# ──────────────────────────────────────────

class Noeud:
    """
    Représente un nœud dans l'arbre binaire de recherche.

    Chaque nœud stocke un prix (la clé de l'ABR) et la liste
    de tous les joueurs qui ont misé ce prix. On garde une liste
    plutôt qu'un seul joueur parce que plusieurs personnes peuvent
    choisir le même prix — et dans ce cas personne ne gagne avec
    ce prix-là, vu qu'il n'est plus unique.

    Attributs
    ----------
    prix : int
        Le prix misé (entier >= 0).
    joueurs : list
        Liste des noms des joueurs ayant misé ce prix.
    gauche : Noeud ou None
        Sous-arbre contenant les prix strictement inférieurs.
    droite : Noeud ou None
        Sous-arbre contenant les prix strictement supérieurs.
    """

    def __init__(self, prix, joueur):
        """
        Crée un nouveau nœud avec un premier joueur.

        Paramètres
        ----------
        prix : int
            Le prix misé.
        joueur : str
            Le nom du joueur qui vient de miser.
        """
        self.prix = prix
        self.joueurs = [joueur]
        self.gauche = None
        self.droite = None


class ABR:
    """
    Arbre Binaire de Recherche pour gérer les mises de l'enchère LowBid.

    On a choisi un ABR parce qu'il garde les prix dans l'ordre tout seul
    à chaque insertion. Ça veut dire qu'un simple parcours infixe suffit
    pour avoir les prix du plus petit au plus grand, sans avoir besoin
    de trier quoi que ce soit après coup.

    Chaque nœud représente un prix. Si deux joueurs misent le même prix,
    ils sont regroupés dans la même liste — et ce prix ne peut pas gagner.
    """

    def __init__(self):
        """Crée un arbre vide, prêt à recevoir des mises."""
        self.racine = None

    def inserer(self, prix, joueur):
        """
        Ajoute la mise d'un joueur dans l'arbre.

        Si quelqu'un a déjà misé ce prix, on ajoute juste le nouveau joueur
        à la liste existante. Sinon on crée un nouveau nœud pour ce prix.

        Paramètres
        ----------
        prix : int
            Le prix proposé (entier >= 0).
        joueur : str
            Le nom du joueur qui mise.
        """
        self.racine = self._inserer(self.racine, prix, joueur)

    def _inserer(self, noeud, prix, joueur):
        """
        La vraie insertion, récursive.

        On descend dans l'arbre en comparant le prix à insérer avec
        celui du nœud courant : à gauche si c'est plus petit, à droite
        si c'est plus grand, et on ajoute à la liste si c'est égal.

        Paramètres
        ----------
        noeud : Noeud ou None
            Le nœud qu'on est en train d'examiner.
        prix : int
            Le prix à insérer.
        joueur : str
            Le joueur associé à ce prix.

        Retourne
        --------
        Noeud
            Le nœud mis à jour, ou un nouveau nœud si on était sur une
            feuille vide.
        """
        if noeud is None:
            return Noeud(prix, joueur)
        if prix == noeud.prix:
            noeud.joueurs.append(joueur)
        elif prix < noeud.prix:
            noeud.gauche = self._inserer(noeud.gauche, prix, joueur)
        else:
            noeud.droite = self._inserer(noeud.droite, prix, joueur)
        return noeud

    def infixe(self):
        """
        Renvoie toutes les mises dans l'ordre croissant des prix.

        C'est le parcours gauche → nœud → droite. Grâce à la structure
        de l'ABR, ça donne automatiquement les prix triés du plus bas
        au plus élevé — exactement ce qu'on veut pour trouver le gagnant.

        Retourne
        --------
        list of tuple
            Liste de (prix, liste_joueurs) dans l'ordre croissant.
        """
        resultat = []
        self._infixe(self.racine, resultat)
        return resultat

    def _infixe(self, noeud, resultat):
        """
        Parcours infixe récursif — remplit la liste au fur et à mesure.

        Paramètres
        ----------
        noeud : Noeud ou None
            Le nœud courant.
        resultat : list
            La liste qu'on remplit avec les tuples (prix, joueurs).
        """
        if noeud is None:
            return
        self._infixe(noeud.gauche, resultat)
        resultat.append((noeud.prix, noeud.joueurs))
        self._infixe(noeud.droite, resultat)

    def afficher(self):
        """
        Affiche toutes les mises dans l'ordre, du prix le plus bas au plus haut.

        Pratique pour voir d'un coup d'œil comment les mises sont réparties
        et vérifier que l'arbre est bien construit.
        """
        print("\n  Récap des mises (du plus bas au plus haut) :")
        print(f"  {'Prix':>6}  {'Nb joueurs':>10}  Qui a misé")
        print("  " + "-" * 45)
        for prix, joueurs in self.infixe():
            print(f"  {prix:>6}  {len(joueurs):>10}  {joueurs}")

    def plus_bas_unique(self):
        """
        Trouve le plus petit prix qu'une seule personne a misé.

        C'est lui qui détermine le gagnant. On profite du fait que
        le parcours infixe donne les prix dans l'ordre : dès qu'on
        tombe sur un prix avec un seul joueur, c'est forcément le plus bas.

        Retourne
        --------
        tuple (int, str) ou None
            (prix, nom_du_gagnant) si quelqu'un gagne, None sinon.
        """
        return self._plus_bas_unique(self.racine)

    def _plus_bas_unique(self, noeud):
        """
        Cherche récursivement le plus bas prix unique.

        On regarde d'abord à gauche (les prix les plus bas), puis le nœud
        courant, puis à droite. On s'arrête dès qu'on trouve un prix
        avec exactement un joueur.

        Paramètres
        ----------
        noeud : Noeud ou None
            Le nœud qu'on examine.

        Retourne
        --------
        tuple ou None
            (prix, joueur) du premier prix unique trouvé, ou None.
        """
        if noeud is None:
            return None
        res = self._plus_bas_unique(noeud.gauche)
        if res is not None:
            return res
        if len(noeud.joueurs) == 1:
            return (noeud.prix, noeud.joueurs[0])
        return self._plus_bas_unique(noeud.droite)

    def successeur(self, prix):
        """
        Trouve le prochain prix juste au-dessus du prix donné.

        Utile par exemple pour savoir quel prix aurait pu gagner si
        le prix courant n'était pas unique.

        Paramètres
        ----------
        prix : int
            Le prix de référence.

        Retourne
        --------
        Noeud ou None
            Le nœud avec le prix juste supérieur, ou None s'il n'existe pas.
        """
        succ = None
        noeud = self.racine
        while noeud is not None:
            if prix < noeud.prix:
                succ = noeud
                noeud = noeud.gauche
            elif prix > noeud.prix:
                noeud = noeud.droite
            else:
                if noeud.droite is not None:
                    tmp = noeud.droite
                    while tmp.gauche is not None:
                        tmp = tmp.gauche
                    succ = tmp
                break
        return succ

    def predecesseur(self, prix):
        """
        Trouve le prix juste en dessous du prix donné.

        Symétrique du successeur, utile pour naviguer dans les mises.

        Paramètres
        ----------
        prix : int
            Le prix de référence.

        Retourne
        --------
        Noeud ou None
            Le nœud avec le prix juste inférieur, ou None s'il n'existe pas.
        """
        pred = None
        noeud = self.racine
        while noeud is not None:
            if prix > noeud.prix:
                pred = noeud
                noeud = noeud.droite
            elif prix < noeud.prix:
                noeud = noeud.gauche
            else:
                if noeud.gauche is not None:
                    tmp = noeud.gauche
                    while tmp.droite is not None:
                        tmp = tmp.droite
                    pred = tmp
                break
        return pred

    def nb_mises(self):
        """
        Compte le nombre total de mises enregistrées.

        On additionne la taille de chaque liste de joueurs dans l'arbre.

        Retourne
        --------
        int
            Le nombre total de mises.
        """
        total = 0
        for _, joueurs in self.infixe():
            total += len(joueurs)
        return total


# ──────────────────────────────────────────
# COÛT ET RECETTE
# ──────────────────────────────────────────

COUT_BASE = 2.0
ALPHA     = 10.0

def cout_mise(prix, cout_base=COUT_BASE, alpha=ALPHA):
    """
    Calcule combien coûte le fait de miser un certain prix.

    La formule : cout_base + alpha / (prix + 1)
    Plus on mise bas, plus c'est cher — c'est la prime de risque.
    Ça évite que tout le monde spam le prix 0 ou 1.

    Paramètres
    ----------
    prix : int
        Le prix qu'on veut miser.
    cout_base : float
        Le coût fixe de base.
    alpha : float
        L'intensité de la pénalité sur les petits prix.

    Retourne
    --------
    float
        Le montant total à payer pour cette mise, en euros.
    """
    return cout_base + alpha / (prix + 1)

def recette_vendeur(abr, cout_base=COUT_BASE, alpha=ALPHA):
    """
    Calcule combien le vendeur a gagné sur une manche.

    Chaque mise est payante, peu importe si le joueur gagne ou pas.
    On additionne les coûts de toutes les mises enregistrées dans l'arbre.

    Paramètres
    ----------
    abr : ABR
        L'arbre avec toutes les mises de la manche.
    cout_base : float
        Coût fixe par mise.
    alpha : float
        Paramètre de la prime de risque.

    Retourne
    --------
    float
        La recette totale arrondie à 2 décimales.
    """
    total = 0.0
    for prix, joueurs in abr.infixe():
        total += len(joueurs) * cout_mise(prix, cout_base, alpha)
    return round(total, 2)


# ──────────────────────────────────────────
# STRATÉGIES DES BOTS
# ──────────────────────────────────────────

def strategie_aleatoire(prix_max=20):
    """
    Le bot mise complètement au hasard.

    Pas de logique particulière, il tire un nombre entre 0 et prix_max.
    C'est la stratégie de référence pour comparer les autres.

    Paramètres
    ----------
    prix_max : int
        Le prix maximum possible.

    Retourne
    --------
    int
        Un prix aléatoire entre 0 et prix_max.
    """
    return random.randint(0, prix_max)

def strategie_prudente():
    """
    Le bot évite les prix trop bas.

    Il préfère miser dans une fourchette moyenne (entre 5 et 15)
    pour ne pas payer trop cher à cause de la prime de risque,
    tout en gardant une chance de gagner.

    Retourne
    --------
    int
        Un prix entre 5 et 15.
    """
    return random.randint(5, 15)

def strategie_agressive():
    """
    Le bot tente sa chance avec des prix très bas.

    Il accepte de payer plus cher à cause de la prime de risque,
    en espérant être le seul à miser dans cette zone.

    Retourne
    --------
    int
        Un prix entre 0 et 5.
    """
    return random.randint(0, 5)


# ──────────────────────────────────────────
# SIMULATION AUTOMATIQUE
# ──────────────────────────────────────────

def simuler_manches(nb_manches=500, nb_joueurs=8,
                    cout_base=COUT_BASE, alpha=ALPHA):
    """
    Simule plein de manches automatiquement pour comparer les stratégies.

    La moitié des joueurs misent aléatoirement, un quart joue prudemment,
    et un quart joue agressivement. À la fin on voit quelle stratégie
    gagne le plus souvent et combien le vendeur empoche en moyenne.

    Paramètres
    ----------
    nb_manches : int
        Combien de manches on simule (500 minimum recommandé).
    nb_joueurs : int
        Combien de joueurs par manche.
    cout_base : float
        Coût fixe par mise.
    alpha : float
        Intensité de la prime de risque.

    Retourne
    --------
    tuple (dict, list)
        - victoires : combien de victoires par stratégie
        - recettes  : la recette du vendeur pour chaque manche
    """
    n_alea = nb_joueurs // 2
    n_prud = nb_joueurs // 4
    n_aggr = nb_joueurs - n_alea - n_prud

    joueurs = []
    for i in range(n_alea):
        joueurs.append(("alea", f"Alea_{i+1}"))
    for i in range(n_prud):
        joueurs.append(("prud", f"Prud_{i+1}"))
    for i in range(n_aggr):
        joueurs.append(("aggr", f"Aggr_{i+1}"))

    victoires  = {"alea": 0, "prud": 0, "aggr": 0}
    recettes   = []
    manches_ok = 0

    for _ in range(nb_manches):
        abr = ABR()
        for strat, nom in joueurs:
            if strat == "alea":
                prix = strategie_aleatoire()
            elif strat == "prud":
                prix = strategie_prudente()
            else:
                prix = strategie_agressive()
            abr.inserer(prix, (strat, nom))

        gagnant = abr.plus_bas_unique()
        if gagnant:
            victoires[gagnant[1][0]] += 1
            manches_ok += 1
        recettes.append(recette_vendeur(abr, cout_base, alpha))

    recette_moy = round(sum(recettes) / nb_manches, 2)

    print(f"\n  Résultats sur {nb_manches} manches :")
    print(f"  Manches avec un gagnant : {manches_ok}/{nb_manches}")
    print(f"\n  Taux de victoire par stratégie :")
    for strat, nb in victoires.items():
        pct = round(nb / nb_manches * 100, 1)
        print(f"    {strat:6} : {nb:4} victoires  ({pct} %)")
    print(f"\n  Recette moyenne du vendeur : {recette_moy} €")
    print(f"  Paramètres : cout_base={cout_base}, alpha={alpha}")

    return victoires, recettes


# ──────────────────────────────────────────
# MODE ENCHÈRE INTERACTIVE
# ──────────────────────────────────────────

def lancer_enchere():
    """
    Lance une vraie partie où les joueurs entrent leurs mises eux-mêmes.

    On demande d'abord combien de personnes veulent jouer, puis chacun
    entre son prénom et son prix. Des bots peuvent aussi être ajoutés
    pour compléter la partie. À la fin on affiche les mises, le gagnant
    et ce que le vendeur a gagné.
    """
    print("\n" + "=" * 50)
    print("   Bienvenue sur LowBid — Qui perd gagne !")
    print("=" * 50)
    print("\n  Le principe : misez le plus bas prix que vous")
    print("  êtes le SEUL à avoir choisi. Bonne chance !\n")

    # combien de joueurs veulent participer
    while True:
        try:
            nb_joueurs = int(input("  Combien de personnes veulent jouer ? "))
            if nb_joueurs < 1:
                print("  Il faut au moins une personne pour jouer !")
                continue
            break
        except ValueError:
            print("  Entre un nombre entier stp.")

    # est-ce qu'on veut des bots ?
    while True:
        try:
            nb_bots = int(input("  Combien de bots voulez-vous ajouter ? (0 si aucun) "))
            if nb_bots < 0:
                print("  Ça peut pas être négatif.")
                continue
            break
        except ValueError:
            print("  Entre un nombre entier stp.")

    abr = ABR()

    # chaque joueur entre son prénom et son prix
    print("\n  Ok, à vous de jouer !\n")
    for i in range(nb_joueurs):
        print(f"  --- Joueur {i+1} ---")
        nom = input("  Ton prénom : ").strip()
        if not nom:
            nom = f"Joueur_{i+1}"
        while True:
            try:
                prix = int(input(f"  {nom}, quel prix tu mises ? (>= 0) : "))
                if prix < 0:
                    print("  Le prix doit être 0 ou plus.")
                    continue
                break
            except ValueError:
                print("  Entre un entier stp.")
        abr.inserer(prix, nom)
        print(f"  Mise enregistrée ! (ça t'a coûté {cout_mise(prix):.2f} €)\n")

    # les bots misent tout seuls
    if nb_bots > 0:
        print("  --- Les bots misent... ---")
        for i in range(nb_bots):
            prix_bot = strategie_aleatoire()
            nom_bot  = f"Bot_{i+1}"
            abr.inserer(prix_bot, nom_bot)
            print(f"  {nom_bot} a misé : {prix_bot}")

    # on affiche toutes les mises
    abr.afficher()

    # le verdict
    print("\n  --- Et le gagnant est... ---")
    gagnant = abr.plus_bas_unique()
    if gagnant:
        print(f"\n   {gagnant[1]} gagne avec le prix {gagnant[0]} !")
    else:
        print("\n  Personne ne gagne cette manche — aucun prix n'était unique.")

    print(f"\n  Recette du vendeur : {recette_vendeur(abr)} €")
    print(f"  Nombre total de mises : {abr.nb_mises()}")

    # un peu d'analyse autour du prix gagnant
    if gagnant:
        p = gagnant[0]
        succ = abr.successeur(p)
        pred = abr.predecesseur(p)
        print(f"\n  Analyse autour du prix gagnant ({p}) :")
        print(f"    Prix juste au-dessus : {succ.prix if succ else 'aucun'}")
        print(f"    Prix juste en dessous : {pred.prix if pred else 'aucun'}")

    print("\n" + "=" * 50)


# ──────────────────────────────────────────
# LANCEMENT
# ──────────────────────────────────────────

# simulation automatique pour comparer les stratégies
print("Simulation automatique en cours...\n")

print("--- 500 manches, paramètres normaux ---")
simuler_manches(nb_manches=500, nb_joueurs=8)

print("\n--- 300 manches, prime de risque faible (alpha=2) ---")
simuler_manches(nb_manches=300, alpha=2.0)

print("\n--- 300 manches, prime de risque forte (alpha=50) ---")
simuler_manches(nb_manches=300, alpha=50.0)

# enchère interactive — les joueurs entrent leurs mises
print("\n")
lancer_enchere()


#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOO0000000000000000000000000000000OOdoodooooooooddddooollccllllllldddolllllcllolddddoooolloooddddooddoddddddoolloollllclcccccc
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOO0000000KKK0000000000000000000000kddooddooddooddddooodolllooooooddooollollollldddxkkxdolloddddodddodddddddolllllc:::cccllcccc
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOO000000000KK00000000000000KKKKKK00kxoollodolooodddddddddooooooddooooooloolllolcldddxxxddllloddddddooodoooddoolccccccccccccldooo
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOO00000000000000000000000000KKKKKK0kxddooodddddodddooodddoodddddoooddlllodxdloodkxokOxxxxxdoodddddoolooolclooolccccclxlllccc:coxx
kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOO0000000KKKK000000000000KKKKKKKK0OxdddoloodddodoooooddddooxxdoooooodooodxkOdoxKKOoxxxdodooooodoollooddxxdollccoolcclxolollodxxx
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOO000000000KKKK0000000000KKKKKKKKK0Oxooddddodododdddoddddddoddddoodldoldx0KKXKkd0KKOodddddooolllododdxddddollcccllcc:cldooddxkkkx
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOO00000000KKKKKK0000000000KKKKKKKK0kdddoooooooooooddoodddddoodooooooddoxk0KKXKKxokKKoodoooollolldoddddooooolccccccllccllxxxxxkxdo
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOkkOOOOOOOOOOOOOOOOOOOOOOO000000000KKKKKK000K000000KKKKKKKKK0kxoddoddododoooooodddddooxxddooddddxk0OOO0KKOkoxdooloooddoooodddoooooodolcccccllloddxxkxddoll
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOO000000KKKKKKKKK000K0000KKKKKKKK0Okddddddoooodooooodddddodkkkkddddoddxxxxdk0Okxxxddoodooooooooooooooooololdoccclllldxkxdoloolod
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOO0000000KKKKKKKKK00K0000KKKKKKKKK0OxxdddddoodooooddddddoodkkO0Okxxxdddddddxkddddododoooooooododoloooodlccccoollllodxkxdodoloodx
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOO000000KKKKKKKK00000000KKKKKKKKKKkxdxxdoodddooddddddddddxdOOOxxddxxddddddddddoooooodddollooooooolllcccclcccoooddxxkxdoooloddxx
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000KKKKKKKKKKKKKK0KKKKK000Okkxxdddddoddodddddddxkkddxdddkxddddooooddddddoooooooodddlloooolllllllcclcccldddodxkxoooooddoxkk
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOkOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000KKKKKKKKKK00000KKKK0OxxddoddodoodddoddddodxxdxxdooddxxkxddddxddddoooodoooooollodoollccccclllllllccldxxkkxoddooooddkOOOO
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000KKKKKKKKKK00000KKKK0OxddooooooooddoodxxdxO00OOkxkloxxxxxxkkxkxdooooooddoloooolloccllooccccoololooddxkkkkkxdoooodxkOOOkO
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOkkOkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO00000000000KKKKKKKKK0000KKK00OkdooooodddooooddodddxxkxO00OxxxxoxkOxoooddoooooooodddoooooollldooolcccccoollodxkkkkkkxdoolooxOOOOOOO
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000KKKKKKKKK0000KKK0kddoddoooooolooooodoodxkOOk0K000OkkooxxddoooollooooodoooddollooloooollcccccoxodxkkkkkxdoooddoxkOOOOOOO
#kkkkkkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000KKKKKKKKK00000KKKKOdoodoodoodllllooolloddodxkkO0kKkkOOdlooooloddooooooooolooooddollollcccccloddkkkkkkkxoloooddkOO00OOOOO
#kkkkkkkkkkkkkkkkkkOOkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO00000000000000KKKKKKKKKK00000K00xdddddxdooodoooooooooddddoxxkdddOkOkxxdodooddoooooooodoooooodoooooolllolllodxkkkkkkxooollodkOOOOOO0OOOO
#kkkkkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000000000000KKKKKKKK000000OkxddoddddxdooddddoooodoodooldxdxdddxxxxdoooolooollloolllodollooooooooccodddddxkkkkxxxdxdoooxkOOOOOOOOOOOOO
#kkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000000KKKKKKKK00000OkddxkdddddxdoloddolllooodddolxoddddddxddoodolododocllllllllllloooollllcccoddxkkxddodxoooodxOOOOOOOOOOOOOOOO
#kkkkkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000000KKKKKKKKKKKKK0koooodkkxdddddoodddoollooooolldoddddddoooooddooododoooollolldoooooolllccooddxkdlooooddddxdkOOOOOOOOOOOOOO0KO
#kkkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO00000000000000000KKKKKKKKKKKKKK00xooodxkxxdddddoddddooloooooodddddolldddoodxoloodododdodllooooooloolclooddxxkdodoooddddxxOOOOOOOOOOOOOO00OOO
#kkkkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000000000KKKKKKKKKKKKKKKK0xdddddddkxxddddodddoodddxddddddddooooxoloooddloodoooodooxxdodollcccoloddxxdxkxoooooddxkOOOOOOOOOOOOO0KOOOOOO
#kkkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000000000000000KKKKKKKKKKKKKKKKK00kxdoddxxddddoodddddddoododolodoodooooddlllcollodddoooooddddolccccccldlxxdodxdooddooxOOOOOOOOOOOOOO00OOOOOOOO
#kkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000000000KKKKKKKKKKKKKKKKKKKKK0kxdooddddddoooodddddoooooooloddddolcloollllooodddoddoooololllloooodoooooooddoooodxkOOOOOOOOOOOOO00OOOOOOOOOO
#kkkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKOxdddddddoddooodoooooocllooddoooooooooododolloooddooololcllllddddxdllllodoooooxOOOOOOkOOOOOO00OOOOOOOOOOOOO
#kkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000000000000KKKKKKKKKKKKKKKKKKKKKKKK0xddxxooodddddddodddooollcodddddoloolldooollodooodolollddoooodxddoolllllooodooxOOOOkOOOOkkOO00OOOOOOOOOOOOOOO
#kkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO00000000000000000000KKKKKKKKKKKKKK00KKKKKKKK0kdxxxddooddddoooodoodddddddodddodooooooooddoodoololollodooloodxdllloooodxxdkkOOOOOOkOkkkO0OOOOOOOOOOO000OOOO
#kkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKK0xdoooddoooooodooodooooddddddddooooooodoodoodooolloolllloddddxxollldoodkxkOOOOOOOOOOOOO00OOOOOOOOOOOO00000000
#kkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKOxdkdooooooooooddddoodddoddodooollldolooolloooolclllollodxxdooooodddkkOOO0OOOOOOOOOO00OOOOOOOOOOOO000000000K
#kkkkkkkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOO000000000000000000000000KK0KKKKKKKKKKKKKKKKKKKKKKKKK0xdddxdddxddddddoododdddddddoooddddoloolllccllllllllolodxxolcloolox0KOOOOOOOOOOOOO0OOOOOOOOOOOOO00000000KKKK
#kkkkkkkkkkkOkOOOOOOOOOOOOOOOOOOOOOOOOOOOO00000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKOOkxxddddddddxxxxddoooddoodooooodooollllllcccccclllcccloxxocccloxodxkOOOOOOOOOOOO00OOOOOOOOOOOO00000000KKKKKKK
#kkkkkkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKK0kdddkxxxxxxxooodxdddxdooddooddooddooolllooolllllcccclodoollccloookkOOOOOOOOOOOOKOOOOOOOOOOOOOOO00000KKKKKKKKKK
#kkkkkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKkdddddkkkxdoooodddddddoddooloooooooodocllllllccclclodxxolccooooxkOOOOOOOOOOO0KOOOOOOOOOOOOOOOO0000KKKKKKKKKKKK
#kOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKOOxoooddxxdooooddodddddxdkxolllooolooodooolccc:ccodxddxolododxkOOOOOOOOOOO00OOOOOOOOOOOOOOO0000KKKKKKKKKKKKK0O
#kkkOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKOxxdddodddddlolodddoooodddddooooooloooooooolccooldddxxllododkOOkOOOOOOk00OOOOOOOOO000OOO0000KKKKKKKKKKKKK0kkxk
#OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO0000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKOdoooooooooooooooooooooooooollllloooooooolllcloddlddllloodOOOOOkkkOOO00OOOOOOOOOOO0000000KKKKKKKKKKKKK0OOkkkkO
#OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO00000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKK0ddooolooodoooddddddoddddddooddollooooooolccclllloolloodkOOOOOOOkkO0OOOOOOOOOOOO000000KKKKKKKKKKKKK0OkkkkxkkOK
#OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO00000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKOxdooooodddooooodxxdoodddooodoolllcccllllcooloccoololxkOO0OOOOOO00OOOOOOOOOOOOOO000KKKKKKKKKKKKK0kkkkxdxk000KK
#OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO0OOO0000O00000000000000000KKKKKKKKKKKKKKKKKXKXXXXXKKKKKK0dodddoooodooooxdddooodooooddollolcclolclddlllcllloxOOkkk0KOOO0OOOOOOOOOOOOOO0000KKKKKKKKKKKK0kkkOOxO00000KNMW
#OOOOOOOOOOOOOOOOOOOOOOOO0000000OOOOO000000000000000000000KKKKKKKKKKKKKKKKKKKXXXXXXXKKKXX0xdddooddooddddodddddooooooooodoooollcccloooooddllokkkkkkkkOOK0OOOOOOOOOOOOO0000KKKKKKKKKKKK0kkkO00000000XNWNX00
#OOOOOOOOOOOOOOOOOOOOO0000000000OO000000000000000000000000KKKKKKKKKKKKKKKKKXXXXXXXXKKKKX0kxkxxooddddddddoddodoooooooodooolldcllllodooxdoodkOOkkkkkkO00OOOOOOOOOOOO0000KKKKKKKKKKK0OkxO00000000KNWWNK00000
#OOOOOOOOOOOOOOOOOOOOO00000000OOOOO0000000000000000000000KKKKKKKKKKKKKKKKXXXXXXXXKKKXXXXX0xxxddddoddoooddddoddoollldddolcclcloodkooddoodOOOkkOKKkO0OOOOOOOOOOOOO000KKKKKKKKKKKKOxkO00000000KWMWX000000000
#OOOOOOOOOOOOOOOOOOOOO000000000OOO0000000000000000000000KKKKKKKKKKKKKKKKKXXXXXXKKKXXXXXXK0Oxddxddoddooooooddddoooooooodlllllooxdooooodkkkxxxkkk00OOOOOOOOOOOOO000KKKKKKKKKKKOkkO0000000KXWWNK000000000O0O
#OOOOOOOOOOOOOOOOOOOOO000000000000000000000000000000000KKKKKKKKKKKKKKKKXXXXXKKKKKXXXXXXXXKOdddddddddooooolooooolooooooollloloddooooxkOOkxxddkOOkOOOOOOOO0000000KKKKKKKKK0OkkO0000000KNWWXK0000000000OOOOO
#OOOOOOOOOOOOOOOOOO0OOOOO00000000000000000000000000000KKKKKKKKKKKKKKKKXXXXKKKKKXXXXXXXXXK0xddddoooodddoddodoodoollcccccccldddxxodkOOOOkkkxkOkOOOOOOOOOO0000KKKKKKKKKK0kkk00000000KWWWK000000000000OOOOOOO
#OOOOOOOOOOOOOOOO000OOOOO000000000000000000000000000KKKKKKKKKKKKKKKKKKKXXKKXXXXXXXXXXXXXXKOddddoodoooooooooooodolllcclldoooddkdkOOOOOOkk0OOOOOOOOOOOO0000KKKKKKKKK0kkkO00000KKXWWNK0000000000OOOOOOOOOOOO
#OOOOOOOOOOOO00O000OOOO0000000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKXXXXXXXXXXKKOkkkxooddoooooddooooooodddollodllooddxkOOOOOOOOO0OOOOOOOOOO00000KKKKKKKKK0kkO0000000KNWNXK0000000000OOOOOOOOOOOOOOO
#OOOOOOOOOOOOOOO000000000000000000000000000000000K00KKKKKKKKKKKKKKKKKKKKKXXXXXXXXXXXKxdkxxddddoooooooollldodddolooclllolodkkOOOkOOO0OOOOOOOOOOOO000KKKKKKKKK0kkO0000000XWWX000000000000OOOOOOOOOOOOOOOOOO
#OOOOOOOOOOOOOOO00000000000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKXXXXXXXXXKKXKkkddddddoodooolloooollloollllloollloxkOOOOOOO0OOOOOOOOOO0000KKKKKKKKK0kkO000000KNWWK00000000O00OOOOOOOOOOOOOOOOOOOOOO
#OOOOOOOOOOOOOOO00000000000000000000000000000000KKKKKKKKKKKKKKKKKKKKKXXXXXXXXKKKKKXX0xxxddddoooooodooolloooollloooooooxOOOkOOOO0OOOOOOOOOO0000KKKKKKKK0kkk00000KXWWNK000000000OOOOOOOOOOOOOOOOOOOOOOOOOOO
#OOOOOOOOOOOO0000000000000000000000000000000000KKKKKKKKKKKKKKKKKKKKXXXXXXXKKKKKKKKK0xddooooodoooooodooooolllllollllokOOOOOOk0OOOOOOOOOOO000KKKKKKKKOkkOOO000KNWWX0000000000000OOOOOOOOOOOOOOOOOOOOOOOOOOO
#OOOOOOOOOOOO00000000000000000000000000000K00KKKKKKKKKKKKKKKKKKKKKXXXXKKKKKKKKKKK0OdooooooodoooooooooooolodolccccokOOOOOOO0kOOOOO0OOO000KKKKKKKKOkO000KKKKWMWK0000000000000000OOOOOOOOOOOOOOOOOOOOOOOOOOO
#OOOOOOOOOO0000000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0doodddddooooooooooooddolllllclxOOOOOOOOOOOOOOOOO000KKKKKKKKOkO00000KXNWXK0000000000000000000OOOOOOOOOOOOOOOOOOOOOOOOOOO
#OOOOOO0000000000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKXKKKKKKKKKKKKKKKKKKkoooloooolccloollooodddoollldkOOOOOO0OOOOOOOO0000KKKKKKKKOOO00000KNWNK00000000000000000000000OOO0000000000OOOOOOOOOOOOOO
#000000000000000000000000000000000K0KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0xddodooollcoooddddoooooddookOOOOOO0OOOOOOOO000KKKKKKKKOkO00000KWMNK000000000000000000000000000000000000000OOOOOOOOOOOOOO
#00000000000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0xddxdoooolldxdoodoolloddkOOOOOOOOOOOOO00O00KKKKKK0OkO0000KXNWX00000000000000000O0000000000000000000000000OOOOOOOOOOOOOO
#0000000000000000000000000000K0KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKOdddoodooollcclooooooxOOOOkO0OOOOOO0O00KKKKKKK0OkO0000KNWNK0000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOO
#000000000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKOxdddddoooolccddddkOOOOOOOOOOOOOOO00KKKKKK0OkO0000KNWX00000000000000000000000000000000000000000OOOOOOOOOOOOOOOOOOOOOOO
#000000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0koodoololllcldxOOOkOOOOOOOOO000KKKKKK0OkO0000KNNK00000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOOOOO
#0000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0OOkkkkkOOO0KKKKKKKKKKKKXXKKKkddddoloolllokOOOOOOOOOOOOO000KKKKK0Ok0000KXNNK00000000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOOOOO
#000000000000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK00kxxdddddxxxxxxxxxxkOKKKKKKKKKOxxdddddooooookOOOOOOOOOOOOO00KKKKK0kk0000KWNK00000000000000000000000000000000000000000000000OO0000OOOOOOOOOOOOOOOOOOOOO
#0000000000KKK000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK00OkxxxxxkkkkOOO000000Okkxxx0KKOxdddddddddoodkOOOO0OOOOOOO00KKKKK0Ok0000XWN000000000000000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOOOOOO
#00000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0OkxxxxkkO0KKXXXXXXXXXXXKKKKKK0kxdddddddoddooxkOOOO0OOOOO000KKKKK0OOO00KXNX0000000000000000000000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOO
#00000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0OkkxkxkO0KXXXXXXXKKKKKKKKKKKKKKKKK00OxddoddddxOOkO0OOOOOO000KKKK0OO000KNXK0000000000000000000000000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOO
#000000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0OkkkkO00KKXXXXXXXXKKKKKKKKKKKKKKKKKKKK000kdddxOOOOOOOOOO000KKKK0kO000KWN000000000000000000000000000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOOOO
#0000000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKK00OkxxkO0KKXXXXXXXXKKXXKKKKKKKKKKKKKKKKK0000OOkOOOOOOOOO0000KKK0OO00KXNK0000000000000000000000000000000000000000000000000000000000000000000000000OOOO000OOOOOOOOOOO
#000KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK00OkkkO0KXXXXXXXXXXXXXXKKKKKKKKKKKKKKK00000000OkkkOOOOO000KKK0OO00XNXK000OOOOOOO0000000000000000000000000000000000000000000000000000000000000000000000OOOOOOOOOOOOOO
#0KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK00OkkkOKXXXXXKKKXXKK0OkkkOOOO0KKKKKKKKKKK000000OkkxxO00KKKKOxO00XWK000OOOOOOOOOOO00000000000000000000000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOOOO
#0KKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0OOkxkOKXXXXKKKKK0OOOOOOkxxkOO000KKKKKKKKK0000Okkxdxxx0K0OOO00KNK0000OOOOOOOOOOOOO00000000000000000000000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOOOO
#0KKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0OOOOkxk0KXXXKKKK00KKKXKKKKK000000KKKKKKKKK0000OOkxddddddxk00KXK00OOOOOOOOOOOOOOO0000000000000000000000000000000000000000000000000000000000000000000000OOOOOOOOOOOOOOOOOO
#00KKKKKKKKKKKKKKKKKKKKKKKKKKKKKK00OkkkOKXXXKK00KXXKKKKKKKKKKKKKKKKKKKKXKKK0OkkkkxddoooodddX00OOOOOOOOOOOOOOOOOOO0000OOOOOOO000000000000000000000000000000000000000000000000000000000000000000000000OO000
#00KKKKKKKKKKKKKKKKKKKKKKKKKKKK00Oxxxk0XXXXKK0KKKKKKKKXXKKK0000000KKKKXXXKKOxdddddxkkkkkOOkxxOOOOOOOOOOOOOOOOOOO00000OOOO00000000000000000000000000000000000000000000000000000000000000000000000000000000
#00KKKKKKKKKKKKKKKKKKKKKKKKKKKK0kxxxk0XXXXKKKKKK0OOO0KXXXK0OOOOOO00KKXXKKK00kxdddxkO0000OOkkxxOOOOOOOOOOOOOOOOOO00000000000000000000000000000000000000000000000000000000000000000000000000000000000000OOO
#000KKKKKKKKKKKKKKKKKKKKKKKKKKK0kxkkOKXXXXKKKKKK000KKKKK00OOkkkkkO0KXXKKK00OOkddx0XXXKK0OOkkxdxOOOOOOOOOOOOOOOOOO0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#dO00000KK00KKKKKKKKKKKKKKKKKOxxkkO0KXXXKXXXXXKK000OkkkkkkxkkkOO0KKKKKK0000Okxdox0000OOkxxxxdxOOOOOOOOOOOOOOOOOOO000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#::::loxkO000KKKKKKKKKKKKKKKKKOxxxkO0XXKXXXXXXXKKKKKKKKKKKKKKKKKKKKKKK00KX000OkxddxOOkxddxxxdddOOOOO0OOOOO00000OO0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#:::,,,,,,:clokKKKKKKKKKKKKKKK0xxxkOKKXKKKXXXKKKKKKKKKKKKKKKKKKKKKKKKKK000000kddddxxk000kxdodddkOOOO00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#cc:,,,,::,,,,,,:clx0KKKKKKK00OxdxkO0XXKKKKKKKKKKKKKKKKKKKKKKKKKKKKK0OxdxOOOkdooddxxxxOK000OkxdxOOO000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#,:::::,::,,,,,,:cc::::coxO00Okxddxk0KKKKKKKKKKKKKKKKKKKKKKKKK000000xooldkOOdcclodddxxxk00000OkxOO0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#''',,,,,:::,,,,,,,:::cccclxkkxddddkOKKKKKKKKKKKKKKKKKKKKK000000000OxkO0K00kdocclodddxxxk0000OkkO0000000000000000000000000000000000000KKXXXNNNNNNNWWWNX00000000000000000000000000000000000000000000000000
#''''''''',,,,,::::::::::ccoxxxddxxk0KKKKKKKKKKKKKKKKKK000000KKK000OO0KKKKK0OOkdoddddddxxxOK00OkkO00000000000000000000KKXNNNWWWWWWWWWWWWWWWWWWWWWWWNXK000000000000000000000000000000000000000000000000000
#'''''''''''''',,,,,,::::::clxO00OkxOKK00KKKKKKKKKKKKKKK0000KKKKK0000000KKKKK0OxdoddddxxxxxO00OkxOKKKKKXXXNWWWWWNNNWWWWWWWWWWWWWWWWNNNWWWWWWWNNNNNXK00000000000000000000000000000000000000000000000000000
#''''''''''''''''''',,,,,,,:cdook00Ok0000KKKKKKKKKKKKKKKK0KKKKK000OOOOO00OOOO00OOkkkkxxkkOO00OOkkkNWWWWNNNNNNNNNNNXXXXXXXXXKKKKKKKK0000000000000000000000000000000000000000000000000000000000000000000000
#''''''''''''''''''',''',',,:ooodkKX00KKKKKKKKKKKKXXXKKKKKKKKK00OkkkxxxxkxkOOOOOkkkkkkkO00000OOkxxO000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#''''''''''''''',''',''''',,:ooddxxdxkO0KKKKKKKKKKXXXXKKKKKKK0KOxdoollldxxdxxkkOOkkkkkkkO00000OOkxkO000O0000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#''''''''''''''''''',,,,,,,,cdddddooddddxk0KKKKKKKXXXXXXXXXKK0OdllllookXXXKKXKkxxkkkxxxkkO0000OOkxx000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#llccc:,''''',,,,,,,,,,,,,,,:lddddoodoooodddxOKKKKXXXXXXXKKK0kolloooxdoclloodoooxxkkkkkkkOO00000Okk000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
#llc::::cclllc:,,,,,,,,,,,,,,codddddddoooodddddxOKKKXXXXXXXKOdllloc::::cccccllcclodkOOOOOOOO0000OxO000000000OO00O0OOOOOOOOOOkkkkkxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxkkkkxxkkkkkxxxxxxxxxxxxxxxxxxxxxk
#c:,,,,,,,,::ccclllllc:,,,,,,,cooodddoooooooodddddxkOKKXXXXX0dlccccllooodxO00OOOOkxok000000O0000Oxlooddddddooooooddddddddddddddddddddddxxxxxxxxxxxxxxxxxxxxxddddddddddddddddxxdddddddxddddddddddddddddddd
#,,,,,,,,,,,,,,,,:::ccllllllc:,looooooooooooooodddddddxk0KXXKKOocllooxOKKKXKKKKK0Okkdx00000OOO0OOxdooodddddddddddddddddddxxxxxxxxxxxxxxxxxkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
#::cccccccc::::::::::::::::ccccccloooooloooooooooooddddddddkKK0OOxoxXXNWNNNXXKKX00OOkOOkkkOOOOOOkxooooooodddddddddddddddxxxxxxxxxxxxxxxxxkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
#::,:,,,,,,:::::::::::::,,,,,,,::,,::::cllllooooooooooooododddxkOOOOkkkOOO0000OkkkkkkkxxxxkOOOOkxdl:::::::cccllloooodddxxxxxxxxxxxxxxxxxxxxxxxxkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
#,,,,,,,,,,,,,,:::,,,,,,,,,,,,,,,,,,,,,:lllllllooooooooooooooooooodxOOOkkkkkkkkkkkkxdoododxkkkkxdd:,,,:::::::::::ccooddddddxxxdddkxddddddddddddxxxxxxxkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
#,,,,,,,,,,,,:,,:::c:cll:::,,,,,,,,,,,,cllllllllllloooooooooollllllclldxkkkkkkxdooooooooooddxxxdoo:::cc:,,,,,,,,::coooooddddddddddddddxxxxdddddddxxxdddddxxkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
#,,,::cclok0000Oxdxx00kxxkkdxkkOkxdllc:cllcclllllllllllllllllllllllcccc:cccccccclllloooooooodddoo:,,,,,,,::::::::,,:::cccllooooddddddddddddddddddxxxddddddddxkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
#:lxk000000000OxxOKOxddxkOOOkkkxxOkOOkxclccccccccccclllllllllllooooooolllllllodxxxxxxxxxdddddoooc,,,,,,,,,,,,,,,,:cccccc:::::::ccllloooodddddddddddxxxxxxddddddddxxxxxxxxxkkkkkkkkkkkkkkkkkkkkkkkkkkkkkkk
#00000000000OxxkOkxddkkkkOOkkkkkkkxdkkxclcccccccccccccclllllllooodddddddddddddddxkOOOOOOkkOkxool,,,,,,,,,,,,,,,,,,looooocc:ccccccccc:,,::cllllooddddddddxxdxxxxxxxxxxxxxxxxddxddddxxxkkkkkkkkkkkkkkkkkkkk
#00000000OxxxxxxxxdxkOkOOOkkkkkkkkkkkxoclccccccccccccccclllooooodddddddddddddddddddxxkkkkkkxool:,,,,,,,,,,,,,,,,,,cooodddddddoolccccccccclcc:,,,:ccllloooodddddddddxxxxxxxxxxxxxxxxxxxxxxxdddxxxxxxkkkkkk
#000OOOkxxxxxxxxxxkkkkOOOkkkkkkkkkkkkkocllcc:ccccccccccccllllooooddddddddooooooodddddddddddoll:'''',,,,,,,,,,,,,,,coooodddddddddddxddoolcccccccccccc:::,:::cloooooooddddddddxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
#kkkkxxxxxxxkxdxkOOOkOOkkkkkkkkkkkkkkklcllcc:::::ccccccccllllllooooooooooooooooloooooolllllllc,''''''''''''''',,,,,looooooodddddddddxxxxxxxddlcccccccllllccccc:,,,,,:clollodddddddddxdxxxxxxxxxxxxxxxxxxx
#xxxdl,'''''''''',cx0Okkkkkkkkkkkkkkkklclllcc:::::::cccccccccllllllllooollllllllllllllccccccll:'''''''''''''''''',,loooooooooddddddddddxxxxxxxxxxxdolllclllcccccccllc:,,,',,:lloolodddoooddddddxxxxxxxxxx
#l,'''''''''''''''''ckkkkkkkkkkkkkkkkdcclllllc:::,,,,::ccccccccccccllllllccccccccccccc::ccclllc,''''''''''''''''''':llooooooooodddddddddddddxxxxxxxxxxxxddoolcccccclccccccccc:::,,',,:ccclodooodddddddddd
#''''''''''''''''''''xkkkkkkkkkkkkko::ccclllllcc::,,,,,,,:::::::::::::::ccc:::::::::::::clllllc,''''''''''''''''''''clllloooooooooodddddddddddxxddddxxxxxxxxxxxddollcclllllllllcccccccc:,,'',,,:clooodddd
#'''''''''''''''''''',xOOOOOkkkkko::',cccllllllccc:::,,,,,,,,,,,,,,:,,:::::::::::,::::cllllloll:'''''''''''''''''''''clllllooooooooooddoooddddddddddddddddxxxxxxxxxxxxxdolccccllllllllcccccccc::,,''''',c
#'''''''''''..'''''''''cxOkkkkkl,,,,,,cccclllllllccc:::::,,,,,,,,,,,,,,,,,,,,:::::ccccllllllollc,'''''''''''''''''''''clllllllllloooooooooodddddddddddddddddddxxxxxxxxxxxxxxxddolllcclccllclcccccccccc::,
#'''''''........''''''''':dOkl:,'','',cccllllllllllccccc::::::::,,,,,,,,::::::ccccccclllllllollc,,,,,''''''''''''''''',ccccllllllllllooooooodddddddddddddddddddddxxdxddxxxxxxxxxxxxxdddllccllllcccccccccc
#...............'''''''''''',,'','''',:ccclllllllllccccccccc:cc::::::::::cccccccccccllllllllolll,'','''''''''''''''''''':cccclllllllllllloooooooddoooddddddddddddddddddxxxxxdxxxxxxxxxxxxxxxdoccclcllcclc
#...............'''''''''''''','''''',:cccllllllllllccccccccccccccccccccccccllllccllllllllooolll:'''''''''''''''''''''''',cccccllllllllllllllloooooooooddddddddddddddddddddxxxdxdxxxxxxxxxxxxxxxxxxdolllc
#...............''''''''''''''''''''',:ccclllllllllllcccccccccccccccccccccclllllllllllllloooolll:''''''''''''''''''''''''''clccccllllllllllllllloooooooooooodddddddddddddddxxxxxddxxxxxxxxxxxxxxxxxxxxxxx
#...................'........'.'''''',:ccclllllllllllllcccccccccccccccccccccccllllllllllllllllll:''',,,,,'''''''''''''''''',::clccclllllllllllllloooooolloooooodddddddddddddddxxxxddxxxxxxxxxxxxxxxxxxxxx
#...........................'''''''':::cccclllllllllllllccccccccccccccccccllllllllllllllllllllll:'',,,,,,,,,,'''''',,,,''''''',,,clccllllllllllllllloooolllllloooooodddddddddddddxxxxxdxxxxdxxxxxxxxxxxxx
#........................'''''''''':c:::ccclllllllllllllllcccccllllllllllllllllllllllllllllllllc,''',,::,,,,,,,,,,,,,,,,,,,,''''''',:clllllllllllllllllllllllllllooooooooddddddddddddddxxxxxdxxxxxxxxxxxx
#......................''''''''''':cc:::ccccllllllllllllllcccclllllllllllllllllllllllllllllllll:'''',,::::,,::::,,,,,,,,,,,,,,,,,,''''',cllllllllllllllllllllllllllooooooooodddddddddxdxdxxdxxxxxxxxxxxxx
#.....................''''''''''',ccc:::cccclllllllllllllcccccccllllllllllllllllllllllllllllllc''''',,::::::::::::::,',,,,,,,,,,,,,,,''''',clllllllllllllllllllolllloooollllooooodddddddddddddxxxxxxxxxxx
#..................'''''''''''''':cccc::ccccccccclllllllccccccccclllllllllllllllcllllllllllllc,''''',,:::::::::::::::,,,,,,,,,,,,,,,,,,''''''':clllllllllllllloooollloooooollooooooooodddddddddddxdxxxxxx
#................''''''''''''''''ccclc:::ccccccccllllllllcccccccccccccccccccccccllllllllllllc''''''',,::::::::::::::::',,,,,,,,,,,,,,,,'''''''''',cllllllllllllooollllooooooooooollooooooooddddddddddxddd
#...............''''''''''''''''':lllcc:::cccccccllllllllllccccccccccccccccccccclllllllllll,''''''''',::::::::::::::::,,,,,,,,,,,,,,,,,,''''''''''''':cccllllllloolllllooooooooollllooooooooooodddddddddd
#..............'''''''''''''''''',clccc:::cccccccccllllllllcccccccccccccccccccccllllllllc:''''''''''',::::::::::::::::,,,,,,,,,,,,,,,,,,'''''''''''''''',,:clllllllllllllooooooolllllllllloollllooooooooo
#.............'''''''''''''''''''',clccc:cccccccccccccllllllccccccccccccccccccccccccllc:,'.'''''''''',::::::::::::::::,,,,,,,,,,,,,,,,,,''''''''''''...''''',:clllllcccccccccc:::::::,,,,,,,,,,,,,,,,,,,,
#..........'''''''''''''''''''''''''':ccc:ccccccccccccccccccccccccccccccccccccccccclcc:'..''''''''''',,:::::::::::::::,''''',,,,,,,,,,,,,''''''''''''.......'''',,,''''''''''''''''''''''''''''''''''''''
#........'''''''''''''''''''''''''''''',:c::cccccccccccccccccccccccccccccccccccccccc:''.'''''''''''''',::::::,,,:::::,,'''''''',,,,,,,,,,,,'''''''''''.........''''''''''''''''''''''''''''''''''''''''''
#......'''''''''''''''''''''''''''''''..',::ccccccccccccccccccccccc::::::::::ccccc:,'.'''''''''''''''',,,::::,,,:::,,,,''''''''',',,,,,,,,,''''''''''''...........'''''''''''''''''''''''''''''''''''''''
#....''''''''''''''''''''''''''''''''''''.',::ccccccccccccccccccccc::::::::::ccc:,'''''''''''''''''''',,,,:::,,,,,,,,,,,'''''''''''''''',,,''''''''''''.............'''''''''''''''''''''''''''''''''''''
#'''''''''''''''''''''''''''''''''''''''''.'',::c:cc:::::cccccccccc:::::::::::::,'''''''..'''''''''''',,,,,,,,,,,,,,,,,'''''''''''''''''''''''''''''''''...............''''''''''''''''''''''''''''''''''
#''''''''''''''''''''''''''''''''''''''''''...',:::::::::::ccccccccccccccccc:,''''''''''.'''''''''''''',,,,,,,,,,,,,,,,'''''''''''''''''''''''''''''''''..................'''''''''''''''''''''''''''''''
#''''''''''''''''''''''''''''''''''''''''''''....',::::::::::cccccccccccc:,'''''''''''''''''''''''''''',,,,,,,,,,,,,,,''''''''''''''''''''''''''''''..'''......................''''''''''''''''''''''''''
#'''''''''''''''''''''''''''''''''''''''''''''.....'',:::::::::ccccc:,''''''''''''''''''''''''''''''''',,,,,,''',,,,'''''''''..''''''''''''''''''''....''.........................'''''''''''''''''''''''
#''''''''''''''''''''''''''''''''''''''''''''''''.....'',,,,,,,'''''''''''''''''''''''.''''''''''''''''''''''''''','''''''''''....'''''''''''''''......'''...........................''''''''''''''''''''
#''''''''''''''''''''''''''''''''''''''''''''''''..................''''''''''''''''''..''''''''''''''''''''''''''''''''''''''''''..'''''''..'''''......''..............................''''''''''''''''''
#'''''''''''''''''''''''''''''''''''''''''''''''''...............''''''''''''''''''''..''''''''''''''''''''''''''''''''''''..''''''''''''''..'''........'.................................'''''''''''''''
#''''''''''''''''''''''''''''''''''''''''''''''''...................''''''''''''''''...''''''''''''''''''''''''''''''''''''''..''''''''''''.................................................''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''.....................'''''''''''''''...'''''''''''''''''''''''''''''''''''''''...''''''''''...................................................'''''''''''
#'''''''''''''''''''''''''''''''''''''''''''''''.............'''''''''''''''''''''''...'''''''''''''''''''''''''''''''''''''''.....''''''''...................................,,.................''''''''
#'''''''''''''''''''''''''''''''''.'.'.''''''''..''''''''''''''''''''''''''''''''''....''''''''''''''''''''''''''''''''''''''''.........'''...................................,....................''''''
#'''''''''''''''''''''''''''''''''.......'''''''''''''''''''''''''''''''''''''''''.....'''''''''''''''''''''''''''''''''''''''''......................................................................'''
#''''''''''''''''''''''''''''''''''......'''''''''''''''''''''''''''''''''''''''''.....'''''''''''''''''''''''''''''''''''''.............................................................................
#'''''''''''''''''''''''''''''''''''..'''''''''''''''''''''''''''''''''''''''''''.....'''''''''''''''''''''''''''''''''''''..............................................................................
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''......'''''''''''''''''''''''''''''''''''.'..............................................................................
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''........'''''''''''''''''''''''''''''''....................................................................................
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''..........''......'''''''''''''''''''''''....................................................................................
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''......................''''''''''''''''''''.......................................................................................
#''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''...................................'''''''''''......................................................''''''............................
#''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''......................................''''''''.....................................................''''''''''...........................
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''................................................................................................'''''''''''''''..........................
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''...........................................................................................''''',''''''''''''''''........................
#'''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''''........................................................................................'''''''',,,,,,,,''''''''''''.....................
#''''''.......''''''''''''''''''''''''''''''''''''''''''''''''''........................................................................................'''''''',,,,,,,,'''''''''''''....................
#''''''.......''''''''''''''''''''''''''''''''''''''''''''''''''.........................................................................................''''''''''''''''''''''''''''''..................
#.............'..'''''''''''''''''''''''''''''''''''''''''''''''..........................................................................................'''''''''''''''''''''''''''''''''..............
#.............''..''''''''''''''''''''''''''''''''''''''''''''''............................................................................................''''''''''''''''''''''''''''''''.............
#....................'''''''.......''''''''''''''''''''''''''''................................................................................................''''''''''''''''''''''''''''''''..........
#....................'''''''.......''''''''''''''''''''''''''''........''''''''''''''...........................................................................'''''''''''''''''''''''''''''''''........
#.....................'''''..................''''''''''''''''.......'''''''''''''''''''...........................................................................'''''''''''''''''''''''''''''''''......
#..................................................................'''''''''''''''''.................................................................................''''''''''''''''''''''''''''''''....
#..................................................................''''''''''''''''....................................................................................''''''''''''''''''''''''''''''''..
#.......................................................................'''''''''''..........................................................................................''''''''''''''''''''''''''''
#............................................................................................................................................................................''''''''''''''''''''''''''''
#............................................................................................................................................................................''''''''''''''''''''''''''''
#............................................................................................................................................................................''''''''''''''''''''''''''''
#..............................................................................................................................................................................''''''''''''''''''''''''''
#...................................................................................................................................................................'............''''''''''''''''''''''''
#.................................................................................................................................................................''..............'''''''''''''''''''''''
#..........................................................................................................................................................................''..''''''''''''''''''''''''''
#...................................................................................................................................................................''''''''''...'''''.''''''''''''''''''
#..................................................................................................................................................................'''''''''''''''''''......'''''''''''''
#.................................................................................................................................................................''''''''.''''''''''..........''''''''''
#..................................................................................................................................................................'''''''..''''''''''......'....''''''''
#..................................................................................................................................................................'''''''''''''',,,,'''....''....'''''''





Simulation automatique en cours...

--- 500 manches, paramètres normaux ---

  Résultats sur 500 manches :
  Manches avec un gagnant : 500/500

  Taux de victoire par stratégie :
    alea   :  172 victoires  (34.4 %)
    prud   :   20 victoires  (4.0 %)
    aggr   :  308 victoires  (61.6 %)

  Recette moyenne du vendeur : 33.12 €
  Paramètres : cout_base=2.0, alpha=10.0

--- 300 manches, prime de risque faible (alpha=2) ---

  Résultats sur 300 manches :
  Manches avec un gagnant : 300/300

  Taux de victoire par stratégie :
    alea   :   94 victoires  (31.3 %)
    prud   :    8 victoires  (2.7 %)
    aggr   :  198 victoires  (66.0 %)

  Recette moyenne du vendeur : 19.47 €
  Paramètres : cout_base=2.0, alpha=2.0

--- 300 manches, prime de risque forte (alpha=50) ---

  Résultats sur 300 manches :
  Manches avec un gagnant : 300/300

  Taux de victoire par stratégie :
    alea   :  102 victoires  (34.0 %)
    prud   :   12 victoires  (4.0 %)
    aggr   :  186 victoires  (62.0 %)

  Re